In [ ]:
Practical No:5
Roll no: 46
Title:Use different voting mechanism and Apply AdaBoost (Adaptive 
Boosting), Gradient Tree Boosting (GBM), XGBoost classification 
on Iris dataset and compare the performance of three 
models using different evaluation measures. Dataset Link 
https://www.kaggle.com/datasets/uciml/iris

In [3]:
!pip install xgboost


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import (
    VotingClassifier, RandomForestClassifier,
    AdaBoostClassifier, GradientBoostingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)

In [7]:
df = pd.read_csv("Iris.csv")
df = df.drop(columns=["Id"], errors="ignore")

In [8]:
X = df.drop("Species", axis=1)
y = LabelEncoder().fit_transform(df["Species"])

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y
)

In [10]:
lr = LogisticRegression(max_iter=200)
dt = DecisionTreeClassifier(random_state=42)
nb = GaussianNB()

hard_voting = VotingClassifier(
    estimators=[("lr", lr), ("dt", dt), ("nb", nb)],
    voting="hard"
)

soft_voting = VotingClassifier(
    estimators=[("lr", lr), ("dt", dt), ("nb", nb)],
    voting="soft"
)

In [11]:
adaboost = AdaBoostClassifier(random_state=42)

gbm = GradientBoostingClassifier(random_state=42)

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42
)

models = {
    "Hard Voting": hard_voting,
    "Soft Voting": soft_voting,
    "AdaBoost": adaboost,
    "GBM": gbm,
    "XGBoost": xgb
}


In [12]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "F1-Score": f1_score(
            y_test, y_pred, average="weighted", zero_division=0
        )
    })

    print("\n", name)
    print(classification_report(
        y_test, y_pred,
        target_names=LabelEncoder().fit(df["Species"]).classes_,
        zero_division=0
    ))



 Hard Voting
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30


 Soft Voting
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       0.90      0.90      0.90        10
 Iris-virginica       0.90      0.90      0.90        10

       accuracy                           0.93        30
      macro avg       0.93      0.93      0.93        30
   weighted avg       0.93      0.93      0.93        30


 AdaBoost
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       0.90      0.90      

In [13]:
comparison = pd.DataFrame(results)
print("\nModel Performance Comparison:")
print(comparison.round(3).to_string(index=False))


Model Performance Comparison:
      Model  Accuracy  Precision  Recall  F1-Score
Hard Voting     0.967      0.970   0.967     0.967
Soft Voting     0.933      0.933   0.933     0.933
   AdaBoost     0.933      0.933   0.933     0.933
        GBM     0.967      0.970   0.967     0.967
    XGBoost     0.933      0.933   0.933     0.933
